<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/01_Data_Scientist/advanced/03_time_series.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Time Series Analysis & Forecasting

Time series data is everywhere: stock prices, sales, sensor readings, web traffic. This notebook covers the full progression from classical statistics (ARIMA) to modern deep learning (LSTM).

**Topics:** Decomposition, stationarity, ARIMA, Prophet, LSTM forecasting, evaluation metrics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

# --- Generate realistic synthetic time series ---
n = 365 * 3  # 3 years of daily data
dates = pd.date_range('2021-01-01', periods=n, freq='D')

# Components: trend + seasonality + noise
t = np.arange(n)
trend     = 100 + 0.05 * t
weekly    = 10 * np.sin(2 * np.pi * t / 7)
yearly    = 25 * np.sin(2 * np.pi * t / 365 - np.pi/2)
noise     = np.random.normal(0, 5, n)
sales     = trend + weekly + yearly + noise

ts = pd.Series(sales, index=dates, name='sales')

print(f'Time series: {ts.index[0].date()} to {ts.index[-1].date()}')
print(f'Length: {len(ts)} days')
print(f'Mean: {ts.mean():.1f}, Std: {ts.std():.1f}')
print(f'Min: {ts.min():.1f}, Max: {ts.max():.1f}')

plt.figure(figsize=(14, 4))
ts.plot(lw=1, color='steelblue')
plt.title('Synthetic Daily Sales (3 Years)', fontsize=13)
plt.xlabel('Date'); plt.ylabel('Sales')
plt.tight_layout(); plt.show()

## 1. Decomposition — Trend, Seasonality, Residual

In [ ]:
# Additive decomposition: Y = Trend + Seasonal + Residual
# Use multiplicative when seasonality grows with trend
result = seasonal_decompose(ts, model='additive', period=365)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
result.observed.plot(ax=axes[0], color='steelblue', lw=1);  axes[0].set_ylabel('Observed')
result.trend.plot(ax=axes[1], color='orange', lw=2);        axes[1].set_ylabel('Trend')
result.seasonal.plot(ax=axes[2], color='green', lw=1);      axes[2].set_ylabel('Seasonal')
result.resid.plot(ax=axes[3], color='red', lw=0.8, alpha=0.7); axes[3].set_ylabel('Residual')
plt.suptitle('Seasonal Decomposition (additive)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Rolling statistics for visual stationarity check
rolling_mean = ts.rolling(30).mean()
rolling_std  = ts.rolling(30).std()

plt.figure(figsize=(14, 4))
ts.plot(alpha=0.4, color='steelblue', label='Original')
rolling_mean.plot(color='orange', lw=2, label='30-day rolling mean')
rolling_std.plot(color='red', lw=2, label='30-day rolling std')
plt.legend(); plt.title('Rolling Statistics (Stationarity Visual Check)')
plt.tight_layout(); plt.show()

## 2. Stationarity & ACF/PACF

In [ ]:
def adf_test(series, name=''):
    """Augmented Dickey-Fuller test for stationarity."""
    result = adfuller(series.dropna(), autolag='AIC')
    p = result[1]
    print(f'{name}: ADF statistic={result[0]:.4f}, p-value={p:.4f}', end='')
    print(f'  → {"STATIONARY" if p < 0.05 else "NON-STATIONARY"} (α=0.05)')
    return p < 0.05

print('=== Stationarity Tests ===')
adf_test(ts, 'Original series')

# First difference removes trend
ts_diff1 = ts.diff(1)
adf_test(ts_diff1, '1st difference ')

# Seasonal difference (period=7 for weekly) removes seasonality
ts_diff7 = ts.diff(7)
adf_test(ts_diff7, 'Seasonal diff(7)')

# Combined
ts_diff_combined = ts.diff(1).diff(7)
adf_test(ts_diff_combined, 'diff(1)+diff(7) ')

# ACF and PACF help identify ARIMA(p,d,q) parameters
# Use stationary series (after differencing)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(ts_diff1.dropna(), lags=40, ax=axes[0, 0], title='ACF of diff(1)')
plot_pacf(ts_diff1.dropna(), lags=40, ax=axes[0, 1], title='PACF of diff(1)')
plot_acf(ts_diff_combined.dropna(), lags=40, ax=axes[1, 0], title='ACF of diff(1)+diff(7)')
plot_pacf(ts_diff_combined.dropna(), lags=40, ax=axes[1, 1], title='PACF of diff(1)+diff(7)')

plt.suptitle('ACF & PACF for ARIMA Order Selection', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('\nACF/PACF reading guide:')
print('  ACF cuts off at lag q → MA(q) process')
print('  PACF cuts off at lag p → AR(p) process')
print('  Both tail off slowly → ARMA process')

## 3. ARIMA Forecasting

In [ ]:
# Train/test split — use last 60 days as test
train = ts[:-60]
test  = ts[-60:]

print(f'Train: {len(train)} days, Test: {len(test)} days')

# ARIMA(p, d, q)
# p = AR order (from PACF), d = differencing order, q = MA order (from ACF)
# For our synthetic data: d=1 (one difference), p=1, q=1
model = ARIMA(train, order=(2, 1, 2))
fitted = model.fit()

print('\nARIMA(2,1,2) Summary (key stats):')
print(f'  AIC:  {fitted.aic:.2f}')
print(f'  BIC:  {fitted.bic:.2f}')

# Forecast
forecast_result = fitted.get_forecast(steps=60)
forecast_mean   = forecast_result.predicted_mean
forecast_ci     = forecast_result.conf_int(alpha=0.05)  # 95% CI

# Evaluation
mae  = mean_absolute_error(test, forecast_mean)
rmse = np.sqrt(mean_squared_error(test, forecast_mean))
mape = (np.abs((test.values - forecast_mean.values) / test.values)).mean() * 100

print(f'\nForecast metrics (60-day horizon):')
print(f'  MAE:  {mae:.2f}')
print(f'  RMSE: {rmse:.2f}')
print(f'  MAPE: {mape:.2f}%')

# Plot
plt.figure(figsize=(14, 5))
train[-120:].plot(label='Train (last 120d)', color='steelblue')
test.plot(label='Actual', color='black', lw=2)
forecast_mean.plot(label='ARIMA Forecast', color='orange', lw=2)
plt.fill_between(forecast_ci.index,
                 forecast_ci.iloc[:, 0], forecast_ci.iloc[:, 1],
                 alpha=0.25, color='orange', label='95% CI')
plt.title(f'ARIMA(2,1,2) Forecast | MAE={mae:.1f}, MAPE={mape:.1f}%')
plt.legend(); plt.tight_layout(); plt.show()

# Model diagnostics
fig = fitted.plot_diagnostics(figsize=(14, 8))
plt.suptitle('ARIMA Residual Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Prophet — Scalable Forecasting

In [ ]:
# Prophet: additive model that handles holidays, multiple seasonalities automatically
# Install: pip install prophet

print('Prophet (Meta) Forecasting')
print('=' * 40)
print('Install: pip install prophet')
print()

prophet_code = '''
from prophet import Prophet

# Prophet expects columns 'ds' (date) and 'y' (value)
df_prophet = ts.reset_index()
df_prophet.columns = ['ds', 'y']
train_p = df_prophet.iloc[:-60]
test_p  = df_prophet.iloc[-60:]

model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='additive',  # or 'multiplicative'
    changepoint_prior_scale=0.05, # flexibility of trend (default=0.05)
    seasonality_prior_scale=10,   # strength of seasonality
)

# Add custom seasonality
model.add_seasonality(name='monthly', period=30.5, fourier_order=5)

# Add holidays (US)
from prophet.make_holidays import make_holidays_df
holidays = make_holidays_df(year_list=[2021, 2022, 2023], country='US')
model.holidays = holidays

model.fit(train_p)

# Forecast
future = model.make_future_dataframe(periods=60)
forecast = model.predict(future)

# Components plot (trend + weekly + yearly + monthly)
fig = model.plot_components(forecast)
plt.tight_layout(); plt.show()

# Evaluation
preds = forecast.set_index('ds').loc[test_p['ds'], 'yhat']
mae   = mean_absolute_error(test_p['y'], preds)
mape  = (abs(test_p['y'].values - preds.values) / test_p['y'].values).mean() * 100
print(f"Prophet MAE: {mae:.2f}, MAPE: {mape:.2f}%")
'''
print(prophet_code)

# Simulate what Prophet output looks like using ARIMA forecast
print('Key Prophet features:')
print('  • Automatic changepoint detection (trend breaks)')
print('  • Multiple seasonalities (yearly, weekly, daily, custom)')
print('  • Holiday effects')
print('  • Robust to missing data and outliers')
print('  • Uncertainty intervals via Monte Carlo sampling')

## 5. LSTM for Time Series Forecasting

In [ ]:
# LSTM: captures long-range temporal dependencies
# Use PyTorch (or TensorFlow)

try:
    import torch
    import torch.nn as nn
    PYTORCH_AVAILABLE = True
    print(f'PyTorch {torch.__version__} available')
except ImportError:
    PYTORCH_AVAILABLE = False
    print('PyTorch not installed. Install: pip install torch')

# --- Data preparation ---
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
ts_scaled = scaler.fit_transform(ts.values.reshape(-1, 1)).flatten()

def create_sequences(data, seq_len=30):
    """Create (X, y) sequences for supervised learning."""
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

SEQ_LEN = 30
X_seq, y_seq = create_sequences(ts_scaled, seq_len=SEQ_LEN)

split = len(X_seq) - 60
X_tr, X_te = X_seq[:split], X_seq[split:]
y_tr, y_te = y_seq[:split], y_seq[split:]

print(f'Train sequences: {X_tr.shape}   Test sequences: {X_te.shape}')
print(f'Each sample: {SEQ_LEN} timesteps → predict next 1 step')

if PYTORCH_AVAILABLE:
    # Convert to tensors
    X_tr_t = torch.FloatTensor(X_tr).unsqueeze(-1)  # (N, 30, 1)
    y_tr_t = torch.FloatTensor(y_tr).unsqueeze(-1)  # (N, 1)
    X_te_t = torch.FloatTensor(X_te).unsqueeze(-1)
    y_te_t = torch.FloatTensor(y_te).unsqueeze(-1)

    class LSTMForecaster(nn.Module):
        def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=1):
            super().__init__()
            self.lstm = nn.LSTM(
                input_size, hidden_size, num_layers,
                batch_first=True, dropout=0.2
            )
            self.fc = nn.Linear(hidden_size, output_size)

        def forward(self, x):
            out, _ = self.lstm(x)   # out: (batch, seq, hidden)
            return self.fc(out[:, -1, :])  # take last timestep

    model_lstm = LSTMForecaster()
    optimizer  = torch.optim.Adam(model_lstm.parameters(), lr=0.001)
    criterion  = nn.MSELoss()

    from torch.utils.data import DataLoader, TensorDataset
    dataset = TensorDataset(X_tr_t, y_tr_t)
    loader  = DataLoader(dataset, batch_size=64, shuffle=True)

    train_losses = []
    for epoch in range(30):
        model_lstm.train()
        epoch_loss = 0
        for X_b, y_b in loader:
            optimizer.zero_grad()
            pred = model_lstm(X_b)
            loss = criterion(pred, y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_lstm.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        train_losses.append(epoch_loss / len(loader))
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1:3d}: loss={train_losses[-1]:.6f}')

    # Evaluate
    model_lstm.eval()
    with torch.no_grad():
        lstm_preds_scaled = model_lstm(X_te_t).numpy().flatten()

    lstm_preds = scaler.inverse_transform(lstm_preds_scaled.reshape(-1, 1)).flatten()
    actual     = scaler.inverse_transform(y_te.reshape(-1, 1)).flatten()
    test_dates = ts.index[-60:]

    mae_lstm  = mean_absolute_error(actual, lstm_preds)
    mape_lstm = (np.abs((actual - lstm_preds) / actual)).mean() * 100

    print(f'\nLSTM Forecast: MAE={mae_lstm:.2f}, MAPE={mape_lstm:.2f}%')

    # Compare all models
    plt.figure(figsize=(14, 5))
    plt.plot(test_dates, actual, 'k-', lw=2, label='Actual')
    plt.plot(test_dates, forecast_mean.values, 'b--', lw=2, label=f'ARIMA (MAPE={mape:.1f}%)')
    plt.plot(test_dates, lstm_preds, 'r-', lw=2, label=f'LSTM  (MAPE={mape_lstm:.1f}%)')
    plt.title('ARIMA vs LSTM: 60-Day Forecast')
    plt.legend(); plt.tight_layout(); plt.show()

    plt.figure(figsize=(8, 3))
    plt.plot(train_losses)
    plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
    plt.title('LSTM Training Loss'); plt.tight_layout(); plt.show()

## 6. Forecasting Best Practices

In [ ]:
# Key metrics comparison
print('=== Time Series Evaluation Metrics ===')
print()

y_actual = np.array([100, 110, 95, 120, 115])
y_pred   = np.array([105, 108, 98, 115, 112])

mae  = np.mean(np.abs(y_actual - y_pred))
mse  = np.mean((y_actual - y_pred)**2)
rmse = np.sqrt(mse)
mape = np.mean(np.abs((y_actual - y_pred) / y_actual)) * 100
smape = np.mean(2 * np.abs(y_actual - y_pred) / (np.abs(y_actual) + np.abs(y_pred))) * 100

metrics = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'MAPE (%)': mape, 'sMAPE (%)': smape}
for name, val in metrics.items():
    print(f'  {name:<12} {val:.4f}')

print()
print('Metric guide:')
print('  MAE   — easy to interpret, same units as y')
print('  RMSE  — penalizes large errors more heavily')
print('  MAPE  — percentage error, breaks when y≈0')
print('  sMAPE — symmetric MAPE, bounded [0, 200%]')

print()
print('=== Walk-Forward Validation (proper backtest) ===')
print()

# Walk-forward: retrain on expanding window for each forecast
# Prevents look-ahead bias vs simple train/test split
walk_forward_code = '''
def walk_forward_arima(series, horizon=7, initial_train=200):
    maes = []
    for i in range(initial_train, len(series) - horizon, horizon):
        train_wf = series[:i]
        test_wf  = series[i:i+horizon]
        
        model = ARIMA(train_wf, order=(2,1,2))
        fitted = model.fit()
        forecast = fitted.forecast(steps=horizon)
        
        maes.append(mean_absolute_error(test_wf, forecast))
    
    return np.mean(maes), np.std(maes)

# This gives unbiased estimate of model performance
mean_mae, std_mae = walk_forward_arima(ts)
print(f"Walk-forward ARIMA: MAE = {mean_mae:.2f} ± {std_mae:.2f}")
'''
print(walk_forward_code)

print('Model selection guide:')
print('  ARIMA:   Short series, seasonal data, interpretable')
print('  Prophet: Business time series, holidays, multiple seasonalities')
print('  LSTM:    Long sequences, complex patterns, enough data (>1000 pts)')
print('  TFT:     State-of-art neural, handles covariates well')
print('  N-BEATS: Pure neural, no feature engineering needed')